# Introdução ao Problema do Nonograma e Prova de Complexidade Computacional

O Nonograma é um puzzle de lógica combinatória sobre uma grade bidimensional de dimensões m×n, onde o objetivo é determinar o estado binário de cada célula (preenchida ou vazia). As restrições do problema são dadas por sequências de inteiros associadas a cada linha e coluna, as quais especificam os comprimentos e a ordem exata de blocos contíguos de células preenchidas. Formalmente, o problema de decisão do Nonograma consiste em determinar se existe ao menos uma matriz binária que satisfaça simultaneamente todas as restrições de consistência local das linhas e das colunas.

Embora instâncias pequenas e comerciais de Nonogramas sejam projetadas para serem resolvidas através de deduções locais simples, a versão generalizada do problema é classificada como NP-difícil (NP-Hard). A prova de sua intratabilidade computacional foi originalmente estabelecida por Ueda e Nagao através de uma redução polinomial a partir do problema Circuit-SAT, que é sabidamente NP-completo.

# Esboço da Prova de NP-Hardness via Redução Polinomial

A demonstração consiste em construir um mapeamento de tempo polinomial O(n) que transforma qualquer circuito booleano arbitrário (composto por variáveis de entrada e portas lógicas) em uma instância equivalente de Nonograma. Para isso, a grade do Nonograma é tratada como um plano de circuitos digitais, onde estruturas geométricas específicas (denominadas gadgets) simulam o comportamento dos componentes eletrônicos:

Gadget de Fio (Wire): Fios são representados por sequências longas de células dispostas em diagonais ou linhas direcionadas. A paridade do deslocamento dos blocos preenchidos propaga o valor lógico através da grade. Uma configuração representa o valor lógico Verdadeiro (1) e a configuração alternativa representa o Falso (0). As pistas numéricas adjacentes forçam a consistência desse sinal ao longo de todo o percurso.

Gadget de Divisão (Splitter): Para permitir que o sinal de uma única variável seja utilizado como entrada em múltiplas portas lógicas, utiliza-se um componente de ramificação. Esse gadget garante, por meio de restrições de colunas compartilhadas, que a configuração de preenchimento do fio de entrada seja replicada identicamente para dois ou mais fios de saída.

Gadget de Portas Lógicas (AND / OR / NOT): As operações booleanas são simuladas na interseção de fios horizontais e verticais. Por exemplo, em uma porta lógica AND, o gadget é projetado de forma que o preenchimento de uma célula crítica de saída só seja geometricamente possível se as pistas das linhas e colunas que representam as duas entradas estiverem ativadas no estado Verdadeiro. Qualquer outra combinação força a linha de saída a assumir o estado Falso.

Gadget de Cruzamento (Crossover): Como os circuitos lógicos podem exigir o cruzamento de conexões e a grade do Nonograma é estritamente bidimensional, a prova introduz um gadget de cruzamento planar. Essa estrutura permite que dois fios independentes se cruzem perpendicularmente sem que o estado lógico de um interfira ou corrompa o estado lógico do outro.

Ao interconectar esses componentes, uma fórmula booleana complexa é inteiramente traduzida em um tabuleiro. A restrição final é aplicada à célula que representa a saída do circuito, forçando-a a ser Verdadeira.

Como a construção dessa grade consome tempo polinomial em relação ao tamanho do circuito original, e dado que o Nonograma resultante possui uma solução válida se, e somente se, o circuito lógico original for satisfatível, conclui-se que o problema do Nonograma é tão difícil quanto o Circuit-SAT. Portanto, o problema é formalmente NP-difícil, justificando o uso de abordagens robustas como o mapeamento para uma CNF (Conjunctive Normal Form) e a subsequente resolução via SAT Solvers, conforme implementado a seguir.

In [3]:
%pip install pycosat
import pycosat

In [4]:
# exemplos nonograma

from typing import TypedDict

class Nonograma(TypedDict):
    linhas: list[list[int]]
    colunas: list[list[int]]

nonograma_simples = {
    "linhas": [
        [1, 1],
        [1, 1, 1],
        [1, 1],
        [1, 1],
        [1],
    ],
    "colunas": [
        [2],
        [1, 1],
        [1, 1],
        [1, 1],
        [2],
    ]
}

In [5]:
from itertools import combinations_with_replacement
from typing import List

class ResolvedorNonograma:
    def __init__(self, nonograma: Nonograma):
        self.m = len(nonograma["linhas"])
        self.n = len(nonograma["colunas"])
        self.restricoes_linhas = nonograma["linhas"]
        self.restricoes_colunas = nonograma["colunas"]

        self.cnf = []
        self.var = self.m * self.n + 1 # próxima variável auxiliar
        self.sol = None

    def x(self, i: int, j: int) -> int:
        return i * self.n + j + 1

    def var_aux(self) -> int:
        v = self.var
        self.var += 1
        return v

    def permutacoes(self, n: int, restricoes: List[int]) -> List[List[bool]]:
        if not restricoes:
            return [[False] * n]

        n_blocos = len(restricoes)
        folga = n - sum(restricoes) - (n_blocos - 1)

        if folga < 0:
            return None

        n_baldes = n_blocos + 1 # número de "baldes" nos quais pode-se distribuir a folga

        permutacoes = []

        # gera combinações de onde cada folga pode cair entre os blocos
        for p in combinations_with_replacement(range(n_baldes), folga):
            qtd_baldes = [p.count(i) for i in range(n_baldes)]

            linha = []
            for i in range(n_blocos):
                linha.extend([False] * qtd_baldes[i]) # folga antes do bloco atual
                linha.extend([True] * restricoes[i]) # bloco atual
                if i < n_blocos - 1:
                    linha.append(False) # espaço entre os blocos

            # folga após o último bloco
            linha.extend([False] * qtd_baldes[-1])

            permutacoes.append(linha)
        return permutacoes

    def gera_cnf_eixo(self, restricoes: List[int], vars: List[int]):
        # gera cnf para linha ou coluna

        n = len(vars)

        if restricoes == []: # linha / coluna sem restrições
            for v in vars:
                self.cnf.append([-v])
            return True

        permutacoes = self.permutacoes(n, restricoes)
        if permutacoes == None:
            return False

        # variáveis auxiliares para a transformação de tseitin
        vars_aux = [self.var_aux() for _ in range(len(permutacoes))]
        self.cnf.append(vars_aux) # raíz da árvore, uma das permutações deve ser verdadeira

        for i, p in enumerate(permutacoes):
            w = vars_aux[i]

            # p (permutação) -> w (cláusula auxiliar)
            # disjunção da negação das variáveis de p v w
            p_entao_w = [w]

            for j in range(n):
                if p[j]:
                    p_entao_w.append(-vars[j])

                    # w (cláusula auxiliar) -> p (permutação)
                    self.cnf.append([-w, vars[j]])
                else:
                    p_entao_w.append(vars[j])

                    # w (cláusula auxiliar) -> p (permutação)
                    self.cnf.append([-w, -vars[j]])

            self.cnf.append(p_entao_w)
        return True

    def gera_cnf(self):
        for i in range(self.m):
            vars_linha = [self.x(i, j) for j in range(self.n)]
            if not self.gera_cnf_eixo(self.restricoes_linhas[i], vars_linha):
                self.sol = "UNSAT"
                self.reset()
                return False

        for j in range(self.n):
            vars_coluna = [self.x(i, j) for i in range(self.m)]
            if not self.gera_cnf_eixo(self.restricoes_colunas[j], vars_coluna):
                self.sol = "UNSAT"
                self.reset()
                return False

        return True

    def resolve(self):
        if not self.gera_cnf():
            self.reset()
            self.cnf = []
            return

        output = pycosat.solve(self.cnf)
        if output == "UNSAT":
            self.sol = "UNSAT"
            return

        output_truncado = output[:self.m * self.n]
        output_bool = [True if var > 0 else False for var in output_truncado]
        self.sol = [output_bool[i : i + self.n] for i in range(0, len(output_bool), self.n)]

    def resultado(self):
        if not self.sol:
            print("chame o resolvedor")
            return

        if isinstance(self.sol, str):
            print(self.sol)
            return

        max_linhas = max((len(r) for r in self.restricoes_linhas), default=0)
        max_colunas = max((len(c) for c in self.restricoes_colunas), default=0)

        # restrições colunas
        for i in range(max_colunas):
            linha_str = " " * (max_linhas * 2) # recuo

            for j in range(self.n):
                restricoes = self.restricoes_colunas[j]
                offset = max_colunas - len(restricoes)

                if i < max_colunas - len(restricoes):
                    linha_str += "  "
                else:
                    linha_str += str(restricoes[i - offset]) + " "

            print(linha_str.rstrip())

        # restrições linhas + grade
        for i, row in enumerate(self.sol):
            restricoes = self.restricoes_linhas[i]

            restricoes_str = [str(x) for x in restricoes]
            padding = [" "] * (max_linhas - len(restricoes))
            linha_str = " ".join(padding + restricoes_str)

            linha_str = linha_str + " " + " ".join("■" if var else "□" for var in row)

            print(linha_str)

    def reset(self):
        self.cnf = []
        self.var = self.m * self.n + 1
        self.sol = None

In [6]:
resolvedor_simples = ResolvedorNonograma(nonograma_simples)

resolvedor_simples.resolve()
resolvedor_simples.resultado()

        1 1 1
      2 1 1 1 2
  1 1 □ ■ □ ■ □
1 1 1 ■ □ ■ □ ■
  1 1 ■ □ □ □ ■
  1 1 □ ■ □ ■ □
    1 □ □ ■ □ □


In [7]:
# Caso de Teste UNSAT.
nonograma_impossivel = {
    "linhas": [
        [3],
        [3],
        [3]
    ],
    "colunas": [
        [],
        [3],
        [3]
    ]
}

resolvedor_unsat = ResolvedorNonograma(nonograma_impossivel)
resolvedor_unsat.resolve()

resolvedor_unsat.resultado()

UNSAT


In [8]:
import time

nonograma_grande = {
    "linhas": [
        [3], [4, 2, 1], [3, 1, 1, 1], [3, 1, 2, 1], [2, 1, 2], [4, 2, 2], [2, 2, 2], [1, 3, 1], [2, 2, 1], [4, 3], [1, 1, 2, 2], [1, 1, 1], [1, 1, 1], [5, 2], [3, 2, 3]
    ],
    "colunas": [
        [2], [3], [1, 1], [1, 1, 2, 1], [2, 2, 1, 1], [2, 6, 2], [1, 2, 2], [1, 3, 2], [3, 2, 5], [2, 1, 1], [1, 1], [4, 2, 1], [2, 2, 2, 2], [1, 1, 2, 5], [2, 5]
    ]
}


tempo_inicio = time.time()

resolvedor_grande = ResolvedorNonograma(nonograma_grande)
resolvedor_grande.resolve()

tempo_fim = time.time()
tempo_total = tempo_fim - tempo_inicio

resolvedor_grande.resultado()

print(f"[+] Tempo total gasto (Modelagem + SAT): {tempo_total:.4f} segundos.")

              1 2               2 1
              1 2 2 1 1 3 2   4 2 1
            1 2 1 6 2 3 2 1 1 2 2 2 2
        2 3 1 1 1 2 2 2 5 1 1 1 2 5 5
      3 □ □ □ □ □ □ □ □ □ □ □ □ ■ ■ ■
  4 2 1 □ □ □ □ □ ■ ■ ■ ■ □ □ ■ ■ □ ■
3 1 1 1 □ □ □ ■ ■ ■ □ □ ■ □ □ ■ □ ■ □
3 1 2 1 ■ ■ ■ □ ■ □ □ ■ ■ □ □ ■ □ □ □
  2 1 2 ■ ■ □ □ □ □ □ ■ □ □ □ ■ ■ □ □
  4 2 2 □ ■ ■ ■ ■ □ □ ■ ■ □ □ □ ■ ■ □
  2 2 2 □ □ □ □ ■ ■ □ □ ■ ■ □ □ □ ■ ■
  1 3 1 □ □ □ □ □ ■ □ □ □ ■ ■ ■ □ □ ■
  2 2 1 □ □ □ □ □ ■ ■ □ □ □ □ ■ ■ □ ■
    4 3 □ □ □ ■ ■ ■ ■ □ □ □ □ □ ■ ■ ■
1 1 2 2 □ □ □ ■ □ ■ □ □ ■ ■ □ □ □ ■ ■
  1 1 1 □ □ □ □ □ ■ □ □ ■ □ □ □ □ ■ □
  1 1 1 □ □ □ □ □ □ ■ □ ■ □ □ □ □ ■ □
    5 2 □ □ □ □ □ ■ ■ ■ ■ ■ □ □ ■ ■ □
  3 2 3 □ □ □ ■ ■ ■ □ ■ ■ □ ■ ■ ■ □ □
[+] Tempo total gasto (Modelagem + SAT): 0.4181 segundos.


In [ ]:
import time

# IMAGEM GRANDE

nonograma_desafio = {
    "linhas": [
        [7], [13], [15], [18], [9, 5, 3], [2, 7, 1], [3, 12], [16], [16], [14],
        [12], [11], [13], [4, 7, 4], [10, 6, 2], [6, 1, 1, 2], [6, 2, 1], [1, 8, 1],
        [2, 4, 3, 1], [7, 1, 1, 1], [10, 2, 5], [1, 7, 1, 1], [1, 7], [10, 2, 1],
        [9, 3], [2, 5, 1], [9, 3], [10, 1], [1, 7, 1], [1, 5, 5], [3, 4, 9], [7, 4, 6],
        [14, 2], [17], [19, 1], [21], [22], [23, 1], [19, 8], [21, 2, 2, 1],
        [9, 12, 7], [2, 4, 15, 4], [1, 4, 12, 2, 3]
    ],
    "colunas": [
        [2], [3], [3, 1], [6], [7], [8], [1, 8], [6, 8], [5, 8, 8], [7, 3, 2, 2, 9],
        [8, 8, 14], [9, 10, 12], [23, 13], [6, 15, 12], [23, 12], [24, 12],
        [13, 4, 2, 12], [17, 2, 11], [2, 9, 2, 14], [9, 3, 3, 13],
        [9, 3, 1, 1, 13], [9, 2, 1, 7, 4], [8, 3, 1, 6, 3], [9, 7, 2], [8, 8],
        [7, 4, 3], [4, 1, 3, 1], [4, 4], [5, 1, 3], [5, 3, 6], [4, 3, 1, 3],
        [5, 4, 1], [4, 1, 4], [2, 2, 3], [2, 2, 2], [2, 3, 2], [1, 2, 1, 1, 1],
        [1, 1, 2, 1, 1], [1, 1, 1, 2], [3, 2, 4], [1, 1]
    ]
}


tempo_inicio = time.time()

resolvedor_desafio = ResolvedorNonograma(nonograma_desafio)
resolvedor_desafio.resolve()

tempo_fim = time.time()
tempo_total = tempo_fim - tempo_inicio

resolvedor_desafio.resultado()

print(f"[+] Tempo total gasto: {tempo_total:.4f} segundos.")